# JS13 Praktikum 1 - ANN Biner Sleep Wise
Memprediksi ada atau tidaknya gangguan tidur menggunakan neural network.

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv'))
data['Sleep Disorder'] = data['Sleep Disorder'].fillna('None')
bp = data['Blood Pressure'].str.split('/', expand=True).astype(float)
data['Systolic BP'], data['Diastolic BP'] = bp[0], bp[1]
data['Has Disorder'] = (data['Sleep Disorder'] != 'None').astype(int)
data = data.drop(columns=['Person ID', 'Blood Pressure', 'Sleep Disorder'])
X, y = data.drop(columns='Has Disorder'), data['Has Disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = [col for col in X.columns if col not in numeric]
preprocess = ColumnTransformer([('numeric', StandardScaler(), numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocess', preprocess), ('ann', MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', max_iter=1000, early_stopping=True, random_state=42))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Akurasi ANN: {accuracy_score(y_test, pred):.2%}')
print(classification_report(y_test, pred, target_names=['Tidak ada gangguan', 'Ada gangguan']))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))
with open('js13pbl_ann_binary.pkl', 'wb') as file: pickle.dump(model, file)

Akurasi ANN: 85.33%
                    precision    recall  f1-score   support

Tidak ada gangguan       0.97      0.77      0.86        44
      Ada gangguan       0.75      0.97      0.85        31

          accuracy                           0.85        75
         macro avg       0.86      0.87      0.85        75
      weighted avg       0.88      0.85      0.85        75

Confusion matrix:
 [[34 10]
 [ 1 30]]
